In [1]:
import pandas as pd
from pathlib import Path
import json
import requests
from bs4 import BeautifulSoup
import csv
from astroquery.jplhorizons import Horizons
import math

In [2]:
pd.reset_option('display.max_rows')
pd.set_option('display.max_columns', None)

In [9]:
CWD = Path().cwd().parent
DATASET_DIR = CWD / 'Datasets'
esa_csv = DATASET_DIR / 'European_Space_Agency_NEO_Designations.csv'

NEO_IDs = []
with esa_csv.open('r') as f:
    esa_data = csv.reader(f)
    next(esa_data) # skip header row
    for row in esa_data:
        NEO_IDs.append(row[0])

print("Total NEO IDs loaded:", len(NEO_IDs))
NEO_IDs[:5]


Total NEO IDs loaded: 39926


['100004 1983VA',
 '100085 1992UY4',
 '100756 1998FM5',
 '100926 1998MQ',
 '10115 1992SK']

In [8]:
# scrape data url is different if NEO_IDs contains a space character in betwen the ids such as 100004 1983VA
# url is https://neo.ssa.esa.int/search-for-asteroids?sum=1&des=[part before space]%20[part after space]  like id 100004 1983VA
# url is https://neo.ssa.esa.int/search-for-asteroids?sum=1&des=[id] like id 2022YL8

json_data = {}
for id in NEO_IDs:
    if ' ' in id:
        print(f"Processing ID with space: {id}")
        break
for id in NEO_IDs:
    if ' ' not in id:
        print(f"Processing ID without space: {id}")
        break


Processing ID with space: 100004 1983VA
Processing ID without space: 1979XB


In [10]:
OUTPUT_FILE = DATASET_DIR / "European_Space_Agency_NEOs_Scraped_Data.csv"

if not OUTPUT_FILE.exists(): OUTPUT_FILE.touch()

# load progress if file exists
if OUTPUT_FILE.exists() and OUTPUT_FILE.stat().st_size > 0:
    df_existing = pd.read_csv(OUTPUT_FILE)
    completed_ids = set(df_existing["ID"].tolist())
    print(f"Loaded {len(completed_ids)} completed IDs.")
else:
    df_existing = pd.DataFrame()
    completed_ids = set()
    print("No previous save found. Starting fresh.")

# columns required
columns = ['ID', 'Absolute_Magnitude', 'Est_Dia_In_Km_Min', 'Est_Dia_In_Km_Max',
           'Close_Approach_Date', 'Relative_Velocity_Km_Per_Hr',
           'Miss_Dist_Kilometers', 'Minimum_Orbit_Intersection',
           'Jupiter_Tisserand_Invariant', 'Epoch_Osculation', 'Eccentricity',
           'Semi_Major_Axis', 'Inclination', 'Asc_Node_Longitude',
           'Orbital_Period', 'Perihelion_Distance', 'Perihelion_Arg',
           'Aphelion_Dist', 'Perihelion_Time', 'Mean_Anomaly', 'Mean_Motion',
           'Hazardous']

all_rows = []

# loop
for neo_id in NEO_IDs:
    if neo_id in completed_ids:
        continue
    print(f"Processing {neo_id} ...")

    if (len(completed_ids)) >= 1200:
        print("Reached 1200 entries, stopping for now.")
        break

    #  data row
    data = {col: None for col in columns}
    data["ID"] = neo_id

    # handle ID formatting
    parts = neo_id.split(" ")
    if len(parts) == 2:
        des = f"{parts[0]}%20{parts[1]}"
        jpl_query = parts[0]  # JPL only needs first part
    else:
        des = parts[0]
        jpl_query = parts[0]

    # 1. ESA MAIN PAGE SCRAPE
    url = f"https://neo.ssa.esa.int/search-for-asteroids?sum=1&des={des}"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    values = [div.get_text(strip=True) for div in soup.find_all("div", class_="simple-list__cell")]
    risk = soup.find_all("span", class_="risk-data__text")

    def get_value(label):
        if label in values:
            idx = values.index(label)
            return values[idx + 1]
        return None

    try:
        data['Epoch_Osculation'] = float(get_value('Epoch')) + 2400000.5
        data['Perihelion_Distance'] = float(get_value('Perihelion (q)'))
        data['Aphelion_Dist'] = float(get_value('Aphelion (Q)'))
        data['Eccentricity'] = float(get_value('Eccentricity (e)'))
        data['Inclination'] = float(get_value('Inclination (i)'))
        data['Orbital_Period'] = float(get_value('Orbit period (P)'))
        data['Minimum_Orbit_Intersection'] = float(get_value('Earth MOID'))
        data['Close_Approach_Date'] = get_value('Date')
        data['Absolute_Magnitude'] = float(get_value('Absolute Magnitude (H)'))
        data['Miss_Dist_Kilometers'] = float(get_value('Nominal distance (from Earth center)'))

        # Diameter conversion m → km
        # diameter have span ID: _NEOSearch_WAR_PSDBportlet_:j_idt21:searchForAsteroidTabs:diameter-value 
        diam_span = soup.find("span", id="_NEOSearch_WAR_PSDBportlet_:j_idt21:searchForAsteroidTabs:diameter-value")

        def parse_diameter(raw):
            if not raw or raw.strip() == "-":
                return None, None

            raw = raw.strip().replace(" ", "")

            # case 1: range (e.g. "600-1300*", "600-1300")
            if "-" in raw:
                try:
                    part1, part2 = raw.split("-")
                    part2 = part2.replace("*", "")  # drop "*"
                    min_km = float(part1) / 1000
                    max_km = float(part2) / 1000
                    return min_km, max_km
                except:
                    return None, None

            # case 2: single value (e.g. "1520", "1300*")
            try:
                raw = raw.replace("*", "")
                km = float(raw) / 1000
                return km, km
            except:
                return None, None


        if diam_span:
            diameter_value = diam_span.get_text(strip=True)
            dmin, dmax = parse_diameter(diameter_value)
            data['Est_Dia_In_Km_Min'] = dmin
            data['Est_Dia_In_Km_Max'] = dmax
        else:
            data['Est_Dia_In_Km_Min'] = None
            data['Est_Dia_In_Km_Max'] = None

        

        # Hazardous?
        hazard_text = risk[0].get_text(strip=True) if risk else "Object is not in risk list"
        data['Hazardous'] = "False" if hazard_text == "Object is not in risk list" else "True"

    except Exception as e:
        print(f"ESA main data missing for {neo_id}: {e}")

    # 2. ESA ORBITAL PROPERTIES PAGE
    url1 = f"https://neo.ssa.esa.int/search-for-asteroids?tab=orbprop&des={des}"
    response1 = requests.get(url1)
    soup1 = BeautifulSoup(response1.text, "html.parser")

    def extract_orbital_property(label, soup):
        blocks = soup.find_all("div", class_="row property-group")
        for block in blocks:
            title = block.find("div", class_="col-lg-3 font-weight-bold d-none d-lg-block")
            if title and label in title.get_text(strip=True):
                val = block.find("div", class_="col-sm-10 col-lg-12")
                return val.get_text(strip=True) if val else None
        return None

    try:
        data['Semi_Major_Axis'] = float(extract_orbital_property("Semimajor Axis", soup1) or 0)
        data['Mean_Anomaly'] = float(extract_orbital_property("Mean Anomaly", soup1) or 0)
        data['Asc_Node_Longitude'] = float(extract_orbital_property("Ascending Node", soup1) or 0)
        data['Perihelion_Arg'] = float(extract_orbital_property("Perihelion", soup1) or 0)
        data['Mean_Motion'] = 360.0 / data['Orbital_Period'] if data['Orbital_Period'] else None

        # Jupiter Tisserand Invariant
        a = data['Semi_Major_Axis']
        e = data['Eccentricity']
        i = math.radians(data['Inclination'])
        a_j = 5.204
        data['Jupiter_Tisserand_Invariant'] = a_j / a + 2 * math.cos(i) * math.sqrt((a / a_j) * (1 - e**2))

    except Exception as e:
        print(f"ESA orbital missing for {neo_id}: {e}")

    # 3. NASA JPL DATA
    url2 = (
        f"https://ssd-api.jpl.nasa.gov/sbdb.api?alt-des=1&alt-orbits=1&ca-data=1&ca-time=both"
        f"&ca-tunc=both&cd-epoch=1&cd-tp=1&full-prec=1&phys-par=1&sstr={jpl_query}"
    )

    try:
        response2 = requests.get(url2)
        json2 = response2.json()

        # Perihelion Time (JD)
        for elem in json2.get("orbit", {}).get("elements", []):
            if elem.get("name") == "tp":
                data['Perihelion_Time'] = float(elem.get("value"))
                break

        # Relative velocity
        for ca in json2.get("ca_data", []):
            if ca.get("body") == "Earth":
                v_rel_kms = float(ca.get("v_rel"))
                data['Relative_Velocity_Km_Per_Hr'] = v_rel_kms * 3600
                break

    except Exception as e:
        print(f"JPL data fetch failed for {neo_id}: {e}")

    df_existing = pd.concat([df_existing, pd.DataFrame([data])], ignore_index=True)
    df_existing.to_csv(OUTPUT_FILE, index=False)

    print(f"Saved Object ID '{neo_id}'s data")




print("All IDs processed. Data saved to ", OUTPUT_FILE)


Loaded 1237 completed IDs.
Processing 2002PM203 ...
Reached 1200 entries, stopping for now.
All IDs processed. Data saved to  c:\Users\ismai\OneDrive\Documents\#Rutgers\Data Management\Datasets\European_Space_Agency_NEOs_Scraped_Data.csv


<span style="color:red">**I limited Webscraping to 1200 rows, as it was taking a long time to scrape**</span>

<span style="color:red">**Scraping these 1200 rows of data from the European space agency's webstie took around 4-5 hours**</span>